# Notebook 03: Health Trajectory Analysis Engine
Demonstrates longitudinal trajectory analysis: timeline building, delta calculation, monthly rate of change, moving averages, ADA clinical trend categorization, and overall trajectory classification.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import config
from preprocessing.load_data import load_raw_data
from preprocessing.pipeline import PreprocessingPipeline
from trajectory.timeline import build_patient_timeline
from trajectory.changes import compute_biomarker_changes
from trajectory.moving_average import compute_moving_averages
from trajectory.trends import classify_biomarker_trends
from trajectory.pipeline import TrajectoryPipeline

## 1. Execute Trajectory Engine on Patient Encounters

In [ ]:
raw_df = load_raw_data(source="auto", num_patients=150)
prep = PreprocessingPipeline()
clean_df = prep.fit_transform_initial(raw_df)

traj_engine = TrajectoryPipeline()
traj_df = traj_engine.run(clean_df)
traj_df[["subject_id", "chartdate", "hba1c", "hba1c_change", "hba1c_rate", "hba1c_moving_avg", "hba1c_trend", "overall_trajectory_status"]].head(10)

## 2. Trajectory Distribution Across the Cohort

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x="overall_trajectory_status", data=traj_df, palette="Set2")
plt.title("Overall Trajectory Status Distribution")
plt.xlabel("Longitudinal Progression Status")
plt.ylabel("Observation Count")
plt.show()

## 3. Visualizing Trajectories for Different Clinical Phenotypes

In [ ]:
# Pick a deteriorating patient and an improving/stable patient
sample_det = traj_df[traj_df["overall_trajectory_status"] == "rapidly_deteriorating"]["subject_id"].iloc[0]
sample_sta = traj_df[traj_df["overall_trajectory_status"] == "stable"]["subject_id"].iloc[0]

fig, axs = plt.subplots(1, 2, figsize=(15, 5))

for idx, (sid, title) in enumerate([(sample_det, "Rapidly Deteriorating Patient"), (sample_sta, "Stable Patient")]):
    sub = traj_df[traj_df["subject_id"] == sid].sort_values("visit_number")
    axs[idx].plot(sub["visit_number"], sub["hba1c"], marker="o", label="HbA1c (%)", color="#dc3545" if "Det" in title else "#28a745", linewidth=2)
    axs[idx].plot(sub["visit_number"], sub["hba1c_moving_avg"], linestyle="--", label="3-Visit Moving Avg", color="#6c757d")
    axs[idx].axhline(7.0, color="orange", linestyle=":", label="ADA Target (7.0%)")
    axs[idx].set_title(f"{title} ({sid})")
    axs[idx].set_xlabel("Visit Number")
    axs[idx].set_ylabel("HbA1c (%)")
    axs[idx].legend()

plt.tight_layout()
plt.show()